# Sync Author-Name Curations

Syncs the name half of author curations (set display_name / set full_name) from the users Heroku Postgres database to a local Databricks Delta table. Runs in `jobs/authors.yaml` upstream of `ApplyAuthorNameCurations` and `CreateAuthors`.

The works half lives in a sibling pair in `notebooks/end2end/` — `SyncWorkAuthorCurations.ipynb` + `ApplyWorkAuthorCurations.ipynb` — because its consumers (`MatchAuthors`, `UpdateWorkAuthorships`) live in `walden_end2end.yaml`.

**Sources** (typed views over `openalex_users.public.curations`):
- `author_display_name_curations` — set a display_name for an author profile
- `author_full_name_curations` — set a full_name (used in matching) for an author profile

**Target**:
- `openalex.authors.author_names_curations` — one row per curated author, `(author_id, curated_display_name, curated_full_name, updated_datetime)`

When multiple curators have set the same property for the same author, latest-wins by source `created` timestamp (aggregation handled in the MERGE source query).

No guardrail cell (unlike the works-side sync): the names apply is sticky-write into `openalex.authors.authors` with no DELETE semantics, so a nuked Delta is benign — no production data reverts, just future updates wait until source is fixed.

## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.authors.author_names_curations (
    author_id             BIGINT NOT NULL,
    curated_display_name  STRING,
    curated_full_name     STRING,
    updated_datetime      TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Sync names curations

In [ ]:
%sql
WITH latest_display_name AS (
  SELECT author_id, new_display_name AS curated_display_name
  FROM (
    SELECT
      author_id,
      new_display_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_display_name_curations
  )
  WHERE rn = 1
),
latest_full_name AS (
  SELECT author_id, new_full_name AS curated_full_name
  FROM (
    SELECT
      author_id,
      new_full_name,
      ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_full_name_curations
  )
  WHERE rn = 1
)
SELECT
  COALESCE(d.author_id, f.author_id) AS author_id,
  d.curated_display_name,
  f.curated_full_name
FROM latest_display_name d
FULL OUTER JOIN latest_full_name f USING (author_id)

In [ ]:
%sql
MERGE INTO openalex.authors.author_names_curations AS target
USING (
  WITH latest_display_name AS (
    SELECT author_id, new_display_name AS curated_display_name
    FROM (
      SELECT
        author_id,
        new_display_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_display_name_curations
    )
    WHERE rn = 1
  ),
  latest_full_name AS (
    SELECT author_id, new_full_name AS curated_full_name
    FROM (
      SELECT
        author_id,
        new_full_name,
        ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_full_name_curations
    )
    WHERE rn = 1
  )
  SELECT
    COALESCE(d.author_id, f.author_id) AS author_id,
    d.curated_display_name,
    f.curated_full_name,
    CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest_display_name d
  FULL OUTER JOIN latest_full_name f USING (author_id)
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
SELECT
  COUNT(*)                    AS total_curated_authors,
  COUNT(curated_display_name) AS total_display_names,
  COUNT(curated_full_name)    AS total_full_names,
  MAX(updated_datetime)       AS last_sync
FROM openalex.authors.author_names_curations

In [ ]:
%sql
SELECT * FROM openalex.authors.author_names_curations
ORDER BY updated_datetime DESC
LIMIT 100